In [14]:
import torch
import torch.nn as nn
import torch.functional as F

In [15]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.gpt import GPTModel

In [16]:
GPT_CONFIG_MED = {
    'vocab_size': 50527,
    'context_length': 1024,
    'emb_dim': 1024,
    'n_heads': 16,
    'n_layers': 24,
    'drop_rate': 0.1,
    'qkv_bias': False
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_MED)
model.eval()

GPTModel(
  (tok_emb): Embedding(50527, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    

In [17]:
import tiktoken
from src.gpt import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special = {'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    decoded = tokenizer.decode(flat.tolist())
    return decoded


start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx = text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_MED['context_length']
)

token_ids_to_text(token_ids, tokenizer)

'Every effort moves youmadeliterally them Contents Calculator expl Presumably professionally Aerospacemembers'

### Loss

In [18]:
inputs = torch.tensor([[16833, 3626, 6100], # ["every effort moves",
                       [40, 1107, 588]]) # "I really like"]

targets = torch.tensor([[3626, 6100, 345 ], # [" effort moves you",
                        [1107, 588, 11311]]) # " really like chocolate"]

with torch.no_grad():
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1)
probas.shape

torch.Size([2, 3, 50527])

In [19]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
token_ids

tensor([[[49816],
         [48148],
         [14199]],

        [[32938],
         [33649],
         [41324]]])

In [20]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1:"
      f" {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Jindal Amit Active


In [21]:
text_idx = 0

target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)
text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)

Text 1: tensor([1.6559e-05, 1.3972e-05, 3.0622e-05])
Text 2: tensor([1.3019e-05, 1.5289e-05, 1.1608e-05])


In [22]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
log_probas

tensor([-11.0086, -11.1785, -10.3938, -11.2491, -11.0884, -11.3638])

In [23]:
avg_log_probas = torch.mean(log_probas)
avg_log_probas

tensor(-11.0470)

In [24]:
neg_avg_log_probas = -avg_log_probas
neg_avg_log_probas

tensor(11.0470)

In [25]:
print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)

Logits shape: torch.Size([2, 3, 50527])
Targets shape: torch.Size([2, 3])


In [26]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([6, 50527])
Flattened targets: torch.Size([6])


In [27]:
loss = nn.functional.cross_entropy(logits_flat, targets_flat)
loss

tensor(11.0470)

In [29]:
perplexity = torch.exp(loss)
perplexity

tensor(62757.1953)